# FPF00 — Data Preparation, FingerJetFX Minutiae and Evaluation Protocol (MOLF DB1 / MOLF DB2 / FingerDOS)

**Purpose:** prepare everything the FPF training kernels need, from the raw archives in `FingerPrint-Final/Datasets/`. **CASIA is not used.** Every generated file is written under `FingerPrint-Final/output/` (shared code in `output/00_code/`, data and protocol in `output/FPF00_MOLF_FingerDOS_Data_Prep_FingerJetFX_Minutiae_Subject_Splits_Pairs/`).

| Archive | Dataset | Content | Role |
|---|---|---|---|
| `MOLF_DB1_and_2.7z` | **MOLF DB1** — Lumidigm Venus IP65 (optical, multispectral) | 100 subjects × 10 fingers × 4 impressions = 4,000 PNG images | training, thresholds, validation, held-out source test |
| same archive | **MOLF DB2** — SecuGen Hamster IV (optical) | same 100 subjects and 1,000 fingers, 4,000 PNG images | few-subject embedding centering, held-out sensor-transfer test |
| `FingerDOS_v1.rar` | **FingerDOS** — SecuGen iD-USB SC (optical) | 60 subjects × 6 fingers (thumb, index, middle of both hands) × 10 impressions = 3,600 BMP | independent external test |

**Steps:**
1. Extract both archives to local disk; find the image files (MOLF: `<subject>_<impression>_<finger>.png` in the DB1 / DB2 folders — `.wsq`, `.bmp`, `.tif` are also accepted; FingerDOS: `<LEFT|RIGHT> HAND/<THUMB|INDEX FINGER|MIDDLE FINGER>/<SS><II>.bmp`).
2. Decode every image (PNG and BMP through OpenCV; the NBIS `wsq` codec is installed only if WSQ files are present), check the counts and the identity structure, and index each dataset in a fixed order (subject, hand, finger, impression): this order defines `sample_index` everywhere.
3. **Originals cache:** every decoded image stored losslessly (PNG bytes) with its resolution, so later kernels never need the raw archives.
4. **FingerJetFX minutiae** (same pinned build and ISO 19794-2 parsing as FPM02) at the image resolution (500 ppi).
5. **Standardised images:** 224×224 greyscale, aspect-preserving letterbox on white (identical function to FPM02, the input of the FPM04C image model).
6. **Protocol, 3 reproducible repetitions** (subject-disjoint, the same split for DB1 and DB2):

| Partition | Subjects | Data | Use |
|---|---|---|---|
| train | 60 | 2,400 DB1 images | training, source embedding mean, thresholds (FMR ≤ 1% and EER on clean training scores) |
| validation | 15 | 600 DB1 images | early stopping (validation AUC) and model selection |
| adaptation | 5 (ranked 1–5) | 200 DB2 images | DB2 embedding mean with nested budgets of 1, 2 and 5 subjects (images only, no labels) |
| test | 20 | 800 DB1 + 800 DB2 images | held-out source-sensor test, DB2 test, DB1↔DB2 cross-sensor test |
| FingerDOS | all 60 | 3,600 images | independent external test |

**Pairs (saved per repetition; identical for every method):** within each set all genuine pairs; impostors sampled with a fixed seed, 25% another finger of the **same subject** and 75% a **different subject**, never duplicated. Training pool: 4 impostors per genuine pair (the Siamese loader balances each epoch, as in FPM); validation and test sets: 1 impostor per genuine pair (balanced, as in FPM). Cross-sensor genuine pairs are every DB1 × DB2 impression of the same finger.

**Optional section 10 (standalone):** expanded-impostor pair manifests for the expanded test of FPF01 (traditional pairs are never changed).

**Hardware:** CPU, High-RAM runtime (≈ 30–50 min, resumable: finished parts are skipped).

In [ ]:
%pip install -q openpyxl opencv-python-headless

In [ ]:
# archive tools: 7-Zip for .7z, bsdtar (libarchive) / unrar for .rar
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y p7zip-full libarchive-tools > /dev/null 2>&1; apt-get -qq install -y unrar > /dev/null 2>&1 || true
!which 7z bsdtar unrar || true

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile /content/fpf_lib.py
"""fpf_lib — shared code for the FPF (FingerPrint-Final) kernels: MOLF DB1 training, MOLF DB2 adaptation / sensor transfer, FingerDOS external test.

Every FPF kernel copies this file from <FingerPrint-Final>/output/00_code/ and imports it, so all kernels use the same paths,
protocol loading, metrics, thresholds, figures, workbook format and run log. TensorFlow is NOT imported here (see fpf_tf.py)."""
import hashlib, json, math, os, platform, re, shutil, subprocess, time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

# ============================================================ workspace
DRIVE = Path('/content/drive/MyDrive')
FINAL_FOLDER_NAME = 'FingerPrint-Final'                       # Datasets/ (raw archives, read-only) and output/ (everything generated)
ARCHIVES = {'MOLF': 'MOLF_DB1_and_2.7z', 'FingerDOS': 'FingerDOS_v1.rar'}
DATASETS = ('MOLF_DB1', 'MOLF_DB2', 'FingerDOS')
SENSORS = {'MOLF_DB1': 'Lumidigm Venus IP65 (optical, multispectral)', 'MOLF_DB2': 'SecuGen Hamster IV (optical)', 'FingerDOS': 'SecuGen iD-USB SC (optical)'}
EXPECTED_SAMPLES = {'MOLF_DB1': 4000, 'MOLF_DB2': 4000, 'FingerDOS': 3600}
KERNELS = {'FPF00': 'FPF00_MOLF_FingerDOS_Data_Prep_FingerJetFX_Minutiae_Subject_Splits_Pairs',
           'FPF01': 'FPF01_MobileNetV3Large_Siamese_FullFT_MildAug_DB1Train_FMR1pctThreshold_DB2MeanCentering_FingerDOSTest'}

def find_folder(name, root=DRIVE, max_depth=4):
    """Finds a Drive folder by exact name (breadth-first, depth-limited)."""
    frontier = [Path(root)]
    for _ in range(max_depth + 1):
        nxt = []
        for d in frontier:
            try: children = [c for c in d.iterdir() if c.is_dir()]
            except (PermissionError, OSError): continue
            for c in children:
                if c.name == name: return c
                nxt.append(c)
        frontier = nxt
    return None

def workspace():
    root = find_folder(FINAL_FOLDER_NAME)
    if root is None: raise FileNotFoundError(f"'{FINAL_FOLDER_NAME}' not found under {DRIVE} (add a shortcut to My Drive if it was shared with you)")
    out = root / 'output'; prep = out / KERNELS['FPF00']; data = prep / 'data'
    ws = {'root': root, 'datasets': root / 'Datasets', 'output': out, 'code': out / '00_code', 'prep': prep, 'data': data,
          'index': data / 'index', 'originals': data / 'originals', 'images': data / 'images', 'templates': data / 'fingerjetfx_templates', 'protocol': prep / 'protocol'}
    for k in ('output', 'code'): ws[k].mkdir(parents=True, exist_ok=True)
    return ws

def kernel_dir(ws, key):
    d = ws['output'] / KERNELS[key]; (d / 'figures').mkdir(parents=True, exist_ok=True); (d / 'runs').mkdir(parents=True, exist_ok=True); return d

def copy_verified(src, dst):
    src, dst = Path(src), Path(dst); dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.is_file() and dst.stat().st_size == src.stat().st_size: return dst
    tmp = dst.with_name(dst.name + '.partial'); shutil.copy2(src, tmp); os.replace(tmp, dst)
    if dst.stat().st_size != src.stat().st_size: raise IOError(f'copy size mismatch: {dst}')
    return dst

def local_copy(path, local_dir='/content/fpf_local'):
    path = Path(path); local = Path(local_dir) / path.name; local.parent.mkdir(parents=True, exist_ok=True)
    if not local.is_file() or local.stat().st_size != path.stat().st_size: shutil.copy2(path, local)
    return local

def sha256_of(path, block=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(block), b''): h.update(b)
    return h.hexdigest()

def timer():
    t0 = time.perf_counter(); return lambda: time.perf_counter() - t0

def rel(ws, path): return str(Path(path).relative_to(ws['output']))

# ============================================================ protocol
PARTITIONS = ('train', 'validation', 'adaptation', 'test')
PAIR_SETS = {   # name: (dataset of image a, dataset of image b, role)
    'DB1_train':      ('MOLF_DB1', 'MOLF_DB1', 'training pairs; clean scores give the thresholds (FMR<=1% and EER)'),
    'DB1_validation': ('MOLF_DB1', 'MOLF_DB1', 'early stopping and model selection'),
    'DB1_test':       ('MOLF_DB1', 'MOLF_DB1', 'held-out source-sensor test'),
    'DB2_test':       ('MOLF_DB2', 'MOLF_DB2', 'held-out test on the second sensor (same 20 test subjects)'),
    'DB1xDB2_test':   ('MOLF_DB1', 'MOLF_DB2', 'cross-sensor test: DB1 image vs DB2 image of the test subjects'),
    'FingerDOS_test': ('FingerDOS', 'FingerDOS', 'independent external test (all 60 subjects)')}
TEST_SETS = ('DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test')

def repetitions(ws):
    reps = sorted(p.name for p in ws['protocol'].glob('rep_*_seed_*') if p.is_dir())
    if not reps: raise FileNotFoundError(f"No repetitions in {ws['protocol']}; run FPF00 first.")
    return reps

def rep_seed(rep): return int(rep.split('_seed_')[1])

def load_npz(path):
    with np.load(path, allow_pickle=False) as z: return {k: z[k] for k in z.files}

def load_protocol(ws, rep):
    d = ws['protocol'] / rep; man = json.loads((d / 'protocol_manifest.json').read_text())
    if int(man['repetition_seed']) != rep_seed(rep): raise RuntimeError(f'{rep}: seed mismatch in protocol manifest')
    pairs = {s: load_npz(d / 'pairs' / f'{s}_pairs.npz') for s in PAIR_SETS}
    splits = pd.read_csv(d / 'MOLF_subject_splits.csv')
    adapt = splits[splits.partition == 'adaptation'].sort_values('adaptation_rank').subject_id.astype(str).tolist()
    samples = {ds: pd.read_csv(d / f'{ds}_sample_splits.csv') for ds in DATASETS}
    return {'rep': rep, 'seed': rep_seed(rep), 'pairs': pairs, 'splits': splits, 'adaptation_subjects': adapt, 'samples': samples, 'manifest': man}

def sample_indices(proto, dataset, partition, subjects=None):
    s = proto['samples'][dataset]; m = s.partition == partition
    if subjects is not None: m &= s.subject_id.astype(str).isin([str(x) for x in subjects])
    return s.loc[m, 'sample_index'].to_numpy(np.int64)

# ============================================================ metrics and thresholds
METRIC_COLUMNS = ['roc_auc', 'eer', 'tar_at_fmr_1pct', 'tar_at_fmr_0_1pct', 'fmr', 'fnmr', 'f1_macro', 'balanced_accuracy', 'accuracy', 'f1_genuine', 'precision', 'recall_tar', 'specificity_tnr']
PERCENT_COLUMNS = list(METRIC_COLUMNS)

def eer_threshold(y, score):
    fpr, tpr, thr = roc_curve(y, score); i = int(np.nanargmin(np.abs(fpr - (1 - tpr)))); return float(thr[i])

def fmr_threshold(y, score, target=0.01):
    """Smallest threshold whose false-match rate on these pairs is <= target (accept if score >= threshold)."""
    imp = np.sort(np.asarray(score, np.float64)[np.asarray(y) == 0]); n = len(imp); k = int(math.floor(target * n))
    if n == 0: raise ValueError('no impostor pairs')
    t = np.nextafter(imp[n - k - 1], np.inf)                 # at most k impostors have score >= t (fewer with ties)
    return float(t)

def verification_metrics(y, score, threshold):
    y = np.asarray(y, int); score = np.asarray(score, float); pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    fpr, tpr, thr = roc_curve(y, score); fnr = 1 - tpr; j = int(np.nanargmin(np.abs(fpr - fnr)))
    def tar_at(target):
        ok = np.flatnonzero(fpr <= target); return float(tpr[ok[-1]]) if len(ok) else 0.0
    f1g = 2 * tp / max(2 * tp + fp + fn, 1); f1i = 2 * tn / max(2 * tn + fp + fn, 1)
    return {'threshold': float(threshold), 'roc_auc': float(roc_auc_score(y, score)), 'eer': float((fpr[j] + fnr[j]) / 2), 'eer_threshold_on_this_set': float(thr[j]),
            'tar_at_fmr_1pct': tar_at(1e-2), 'tar_at_fmr_0_1pct': tar_at(1e-3), 'fmr': fp / max(tn + fp, 1), 'fnmr': fn / max(tp + fn, 1),
            'f1_macro': 0.5 * (f1g + f1i), 'f1_genuine': f1g, 'balanced_accuracy': 0.5 * (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)),
            'accuracy': (tp + tn) / max(len(y), 1), 'precision': tp / max(tp + fp, 1), 'recall_tar': tp / max(tp + fn, 1), 'specificity_tnr': tn / max(tn + fp, 1),
            'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'pairs': int(len(y)), 'genuine_pairs': int(y.sum()), 'impostor_pairs': int((1 - y).sum())}

def l2n(E): return E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)
def center(E, mu): return l2n(np.asarray(E, np.float32) - np.asarray(mu, np.float32)[None, :])

def pair_scores(Ea, Eb, pairs):
    return np.sum(Ea[np.asarray(pairs['index_a'])] * Eb[np.asarray(pairs['index_b'])], axis=1)

def to_percent(frame):
    f = frame.copy()
    for c in PERCENT_COLUMNS:
        if c in f.columns: f[c] = 100 * f[c]
    return f

def summarize(results, keys):
    """Mean and std over repetitions (percent) for every group in `keys`, plus 'mean ± std' text columns."""
    pct = to_percent(results); cols = [c for c in METRIC_COLUMNS if c in pct.columns]
    g = pct.groupby(list(keys))[cols]; mean, std, n = g.mean(), g.std(ddof=1), g.size().rename('repetitions')
    out = mean.add_suffix('_mean').join(std.add_suffix('_std')).join(n).reset_index()
    for c in cols: out[f'{c} (mean ± std)'] = [f'{m:.2f} ± {0 if np.isnan(s) else s:.2f}' for m, s in zip(out[f'{c}_mean'], out[f'{c}_std'])]
    return out

# ============================================================ figures (300 dpi PNG)
FIG_DPI = 300
COLORS = plt.get_cmap('tab10').colors

def _save(fig, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight'); plt.close(fig); return path

def fig_roc_det(curves, title, out_dir):
    """curves: {label: (y, score)} -> roc.png and det.png."""
    out_dir = Path(out_dir); paths = []
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    for k, (lab, (y, sc)) in enumerate(curves.items()):
        fpr, tpr, _ = roc_curve(y, sc); ax.plot(fpr, tpr, color=COLORS[k % 10], lw=1.5, label=f'{lab} (AUC {100 * roc_auc_score(y, sc):.2f})')
    ax.plot([0, 1], [0, 1], ':', color='grey', lw=1); ax.set(xlabel='False match rate (FMR)', ylabel='True accept rate (TAR)', title=f'ROC — {title}'); ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=.3)
    paths.append(_save(fig, out_dir / 'roc.png'))
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    for k, (lab, (y, sc)) in enumerate(curves.items()):
        fpr, tpr, _ = roc_curve(y, sc); m = (fpr > 0) & (tpr < 1); ax.plot(fpr[m], 1 - tpr[m], color=COLORS[k % 10], lw=1.5, label=lab)
    ax.axvline(0.01, color='grey', ls='--', lw=1); ax.set(xscale='log', yscale='log', xlabel='FMR', ylabel='FNMR', title=f'DET — {title}'); ax.legend(fontsize=7); ax.grid(alpha=.3, which='both')
    paths.append(_save(fig, out_dir / 'det.png')); return paths

def fig_score_distributions(panels, thresholds, title, path):
    """panels: {label: (y, score)}; thresholds: {name: value} drawn on every panel (frozen training thresholds)."""
    n = len(panels); fig, axs = plt.subplots(1, n, figsize=(4.3 * n, 3.6), squeeze=False)
    for a, (lab, (y, sc)) in zip(axs[0], panels.items()):
        bins = np.linspace(np.min(sc), np.max(sc), 60)
        a.hist(sc[y == 0], bins=bins, alpha=.6, color=COLORS[3], density=True, label='impostor'); a.hist(sc[y == 1], bins=bins, alpha=.6, color=COLORS[0], density=True, label='genuine')
        for k, (tn, tv) in enumerate(thresholds.items()): a.axvline(tv, color=['black', 'grey'][k % 2], ls=['--', ':'][k % 2], lw=1, label=tn)
        a.set(title=lab, xlabel='Cosine score'); a.legend(fontsize=6)
    fig.suptitle(title, fontsize=10); return _save(fig, path)

def fig_training(history, title, path, train_col='verification_auc', val_col='val_verification_auc'):
    h = pd.DataFrame(history); fig, axs = plt.subplots(1, 2, figsize=(11, 3.8))
    if 'loss' in h: axs[0].plot(h.epoch, h['loss'], color=COLORS[0], label='train loss')
    if 'val_loss' in h: axs[0].plot(h.epoch, h['val_loss'], color=COLORS[3], label='validation loss')
    axs[0].set(xlabel='Epoch', ylabel='Loss'); axs[0].legend(fontsize=8); axs[0].grid(alpha=.3)
    for col, lab, c in ((train_col, 'train AUC', COLORS[0]), (val_col, 'validation AUC', COLORS[3])):
        if col in h: axs[1].plot(h.epoch, 100 * h[col], color=c, label=lab)
    axs[1].set(xlabel='Epoch', ylabel='AUC (%)'); axs[1].legend(fontsize=8); axs[1].grid(alpha=.3)
    fig.suptitle(title, fontsize=10); return _save(fig, path)

def fig_mean_roc(curves_by_arm, title, path, grid=np.linspace(0, 1, 501)):
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    for k, (arm, curves) in enumerate(curves_by_arm.items()):
        tprs, aucs = [], []
        for y, sc in curves:
            fpr, tpr, _ = roc_curve(y, sc); tprs.append(np.interp(grid, fpr, tpr)); aucs.append(roc_auc_score(y, sc))
        m, s = np.mean(tprs, 0), np.std(tprs, 0); c = COLORS[k % 10]
        ax.plot(grid, m, color=c, lw=1.5, label=f'{arm} ({100 * np.mean(aucs):.2f}±{100 * np.std(aucs):.2f})'); ax.fill_between(grid, np.clip(m - s, 0, 1), np.clip(m + s, 0, 1), color=c, alpha=.15)
    ax.plot([0, 1], [0, 1], ':', color='grey', lw=1); ax.set(xlabel='FMR', ylabel='TAR', title=title); ax.legend(fontsize=7, loc='lower right'); ax.grid(alpha=.3)
    return _save(fig, path)

# ============================================================ workbook, run log, environment
def env_info():
    info = {'python': platform.python_version(), 'platform': platform.platform(), 'cpu_count': os.cpu_count()}
    try: info['cpu_model'] = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), '')
    except OSError: pass
    try: info['ram_gb'] = round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 1)
    except (ValueError, OSError): pass
    try: info['gpu'] = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True, timeout=10).stdout.strip() or 'none'
    except Exception: info['gpu'] = 'none'
    for mod in ('numpy', 'pandas', 'sklearn', 'tensorflow', 'keras', 'h5py', 'cv2'):
        try: info[mod] = __import__(mod).__version__
        except Exception: pass
    return info

class RunLog:
    def __init__(self, path): self.path = Path(path); self.rows = pd.read_csv(self.path).to_dict('records') if self.path.is_file() else []
    def add(self, **row):
        row = {'time_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), **row}; self.rows.append(row); pd.DataFrame(self.rows).to_csv(self.path, index=False)
        print('[log]', {k: v for k, v in row.items() if k != 'time_utc'})
    def frame(self): return pd.DataFrame(self.rows)

def write_workbook(path, sheets):
    path = Path(path); tmp = path.with_name(path.stem + '.partial.xlsx')
    with pd.ExcelWriter(tmp, engine='openpyxl') as w:
        for name, df in sheets.items():
            if df is None: continue
            if isinstance(df, dict): df = pd.DataFrame([(k, json.dumps(v, default=str) if isinstance(v, (dict, list, tuple)) else v) for k, v in df.items()], columns=['parameter', 'value'])
            df.to_excel(w, sheet_name=str(name)[:31], index=False)
    os.replace(tmp, path); print('Workbook:', path); return path

def figure_index(kernel_root):
    return pd.DataFrame([{'figure': str(p.relative_to(kernel_root)), 'bytes': p.stat().st_size} for p in sorted(Path(kernel_root).rglob('*.png'))])

def run_dir(kdir, rep, arm):
    d = Path(kdir) / 'runs' / rep / arm; d.mkdir(parents=True, exist_ok=True); return d

def run_complete(d): return (Path(d) / 'run_complete.json').is_file()

def mark_complete(d, info):
    (Path(d) / 'run_complete.json').write_text(json.dumps({**info, 'completed_utc': datetime.now(timezone.utc).isoformat(timespec='seconds')}, indent=2, default=str))

def write_winner(kdir, kernel, arm, config, per_rep, selection_table, note=''):
    man = {'kernel': kernel, 'winning_arm': arm, 'config': config, 'selection': 'highest mean MOLF DB1 validation AUC over repetitions (secondary: FMR, FNMR, F1-macro at the training thresholds)',
           'selection_table': selection_table.to_dict('records') if isinstance(selection_table, pd.DataFrame) else selection_table, 'note': note, 'repetitions': per_rep,
           'written_utc': datetime.now(timezone.utc).isoformat(timespec='seconds')}
    (Path(kdir) / 'winner_manifest.json').write_text(json.dumps(man, indent=2, default=str)); print('Winner:', arm); return man

In [ ]:
%%writefile /content/fpf_tf.py
"""fpf_tf — TensorFlow helpers shared by the FPF training kernels (identical to fpm_tf used by FPM04C) (pairs, Siamese model, callbacks, embedding, timing)."""
import math, os, random, time
import numpy as np
import pandas as pd
import tensorflow as tf

def set_all_seeds(seed):
    os.environ['PYTHONHASHSEED'] = str(seed); random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def use_mixed_precision():
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        tf.keras.mixed_precision.set_global_policy('mixed_float16'); return True
    return False

class PairSequence(tf.keras.utils.Sequence):
    """Rotating, balanced pair batches (the protocol's training schedule): each epoch takes `pairs_per_epoch` pairs,
    half genuine, half impostor, impostors 25% same-subject / 75% different-subject, cycling through the whole pool.
    `arrays` is a list of per-sample arrays (concatenated along the last axis); float16 is returned as float32."""
    def __init__(self, arrays, pairs, batch_size, shuffle, seed, pairs_per_epoch=None, same_subject_fraction=0.25, workers=4):
        super().__init__(workers=workers, use_multiprocessing=False, max_queue_size=8)
        self.arrays = arrays; self.pairs = pairs; self.batch_size = int(batch_size); self.shuffle = bool(shuffle)
        self.rng = np.random.default_rng(seed); self.ppe = None if pairs_per_epoch is None else int(pairs_per_epoch); self.ssf = float(same_subject_fraction)
        lab = pairs['label']; nt = pairs.get('negative_type', np.where(lab == 1, 0, 2))
        self.pools = {'positive': np.flatnonzero(lab == 1), 'same': np.flatnonzero((lab == 0) & (nt == 1)), 'diff': np.flatnonzero((lab == 0) & (nt != 1))}
        self.cursors = {k: 0 for k in self.pools}
        if self.shuffle:
            for p in self.pools.values(): self.rng.shuffle(p)
        self._build()
    def _get(self, idx):
        parts = [a[idx] for a in self.arrays]; x = np.concatenate(parts, axis=-1) if len(parts) > 1 else parts[0]
        return x.astype(np.float32) if x.dtype == np.float16 else x
    def __len__(self): return math.ceil(len(self.order) / self.batch_size)
    def __getitem__(self, b):
        pos = self.order[b * self.batch_size:(b + 1) * self.batch_size]
        return (self._get(self.pairs['index_a'][pos]), self._get(self.pairs['index_b'][pos])), self.pairs['label'][pos].astype(np.float32)
    def _take(self, k, n):
        pool = self.pools[k]
        if n <= 0 or len(pool) == 0: return np.empty(0, np.int64)
        if not self.shuffle: return pool[:min(n, len(pool))]
        out = []; r = n
        while r > 0:
            c = self.cursors[k]; t = min(r, len(pool) - c); out.append(pool[c:c + t]); self.cursors[k] += t; r -= t
            if self.cursors[k] >= len(pool): self.rng.shuffle(pool); self.cursors[k] = 0
        return np.concatenate(out)
    def _build(self):
        total = 2 * len(self.pools['positive']); target = total if self.ppe is None else min(self.ppe, total)
        n_pos = target // 2; n_same = int(round(n_pos * self.ssf)) if len(self.pools['same']) else 0
        pos = self._take('positive', n_pos); neg = np.concatenate([self._take('same', n_same), self._take('diff', n_pos - n_same)])
        if self.shuffle: self.rng.shuffle(neg)
        m = min(len(pos), len(neg)); self.order = np.column_stack([pos[:m], neg[:m]]).reshape(-1)
    def on_epoch_end(self):
        if self.shuffle: self._build()

class CosineCalibration(tf.keras.layers.Layer):
    """Turns a cosine into a same-finger probability: sigmoid(scale * cosine + bias). Used only for the BCE loss."""
    def build(self, input_shape):
        self.scale = self.add_weight(name='scale', shape=(), initializer=tf.keras.initializers.Constant(10.0), trainable=True, dtype='float32')
        self.bias = self.add_weight(name='bias', shape=(), initializer=tf.keras.initializers.Constant(-5.0), trainable=True, dtype='float32')
    def call(self, cosine): return tf.nn.sigmoid(tf.cast(cosine, tf.float32) * self.scale + self.bias)

def build_siamese(encoder, input_shape, input_dtype, learning_rate, weight_decay=1e-4, clipnorm=None):
    a = tf.keras.Input(input_shape, dtype=input_dtype, name='input_a'); b = tf.keras.Input(input_shape, dtype=input_dtype, name='input_b')
    p = CosineCalibration(name='same_finger_probability')(tf.keras.layers.Dot(axes=1, name='cosine_similarity')([encoder(a), encoder(b)]))
    model = tf.keras.Model([a, b], p, name='siamese')
    opt = tf.keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=weight_decay, **({'clipnorm': clipnorm} if clipnorm else {}))
    model.compile(optimizer=opt, loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=[tf.keras.metrics.BinaryAccuracy(name='verification_accuracy'), tf.keras.metrics.AUC(name='verification_auc')])
    return model

def callbacks(weights_path, history_csv, patience=10, lr_patience=3, lr_factor=0.3, monitor='val_verification_auc', min_delta=1e-4):
    """Early stopping and learning-rate drops on INTERNAL-VALIDATION AUC (never on calibration, validation or test)."""
    return [tf.keras.callbacks.ModelCheckpoint(str(weights_path), monitor=monitor, mode='max', save_best_only=True, save_weights_only=True, verbose=1),
            tf.keras.callbacks.EarlyStopping(monitor=monitor, mode='max', min_delta=min_delta, patience=patience, restore_best_weights=True, verbose=1),
            tf.keras.callbacks.ReduceLROnPlateau(monitor=monitor, mode='max', min_delta=min_delta, factor=lr_factor, patience=lr_patience, min_lr=1e-7, verbose=1),
            tf.keras.callbacks.CSVLogger(str(history_csv))]

def embed(encoder, x, batch_size=128):
    return encoder.predict(x, batch_size=batch_size, verbose=0).astype(np.float32)

def time_encoder(encoder, batch, repeats=30):
    """Returns (ms per item at this batch size, ms for a single item) on the current device."""
    f = tf.function(lambda z: encoder(z, training=False)); f(batch); t0 = time.perf_counter(); f(batch).numpy(); thr = 1000 * (time.perf_counter() - t0) / len(batch)
    one = batch[:1]; f(one); t0 = time.perf_counter()
    for _ in range(repeats): f(one).numpy()
    return thr, 1000 * (time.perf_counter() - t0) / repeats

def count_params(model):
    tr = int(sum(np.prod(v.shape) for v in model.trainable_weights)); nt = int(sum(np.prod(v.shape) for v in model.non_trainable_weights))
    return {'parameters_total': tr + nt, 'parameters_trainable': tr}

def history_frame(csv_path, extra):
    h = pd.read_csv(csv_path).drop(columns='epoch', errors='ignore'); h.insert(0, 'epoch', np.arange(1, len(h) + 1))
    for k, v in extra.items(): h.insert(0, k, v)
    return h

def fit_finished(history_csv, epochs, patience, monitor='val_verification_auc', min_delta=1e-4):
    """True when a CSVLogger history shows the fit ended (all epochs, or early stopping would have triggered)."""
    try: h = pd.read_csv(history_csv)
    except Exception: return False
    if len(h) >= epochs: return True
    if monitor not in h or len(h) == 0: return False
    best = int(np.argmax(h[monitor].to_numpy())); return len(h) - 1 - best >= patience

In [ ]:
%%writefile /content/fpf_eval.py
"""fpf_eval — optional evaluation modules for the FPF kernels (pure NumPy / pandas / scikit-learn / SciPy; no TensorFlow).

Module A — expanded-impostor verification test: deterministic impostor pools (default 50,000 unique negatives per test set and
           repetition) that contain the traditional genuine pairs unchanged and every traditional impostor, keep the predefined
           25% same-subject / 75% different-subject mixture where feasible, and are scored with the frozen FPF01 thresholds.
Module B — subject-cluster bootstrap: people are resampled with replacement; pair weights follow the subject-multiplicity rule
           (genuine and same-subject pairs: m_i; different-subject pairs: m_i * m_j); thresholds, centering vectors and scores fixed.

Nothing here writes into the traditional artifacts; callers pass explicit output directories."""
import hashlib, json, math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import roc_auc_score, roc_curve

EVAL_VERSION = 'fpf_eval_v1'
TEST_SETS = ('DB1_test', 'DB2_test', 'DB1xDB2_test', 'FingerDOS_test')
TEST_SET_DATASETS = {'DB1_test': ('MOLF_DB1', 'MOLF_DB1'), 'DB2_test': ('MOLF_DB2', 'MOLF_DB2'), 'DB1xDB2_test': ('MOLF_DB1', 'MOLF_DB2'), 'FingerDOS_test': ('FingerDOS', 'FingerDOS')}
STRATUM_NAMES = {0: 'genuine', 1: 'same_subject_different_finger', 2: 'different_subject'}
JOIN_KEYS = ['repetition', 'arm', 'pair_set', 'evaluation_protocol', 'scoring', 'threshold_rule', 'threshold_family', 'metric']
MIXTURE_NOTE = ('combined FMR = FMR over the predefined impostor mixture of this pool (target 25% same-subject/different-finger, '
                '75% different-subject); the realised proportions are reported with every row')

# ============================================================ hashing / provenance
def sha256_file(path, block=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(block), b''): h.update(b)
    return h.hexdigest()

def json_hash(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()

def array_hash(*arrays):
    h = hashlib.sha256()
    for a in arrays: a = np.ascontiguousarray(a); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes())
    return h.hexdigest()

def protocol_hashes(pdir):
    """sha256 of every traditional protocol file of one repetition (subject splits, sample splits, traditional pairs, manifest)."""
    pdir = Path(pdir); files = sorted([*pdir.glob('*.csv'), pdir / 'protocol_manifest.json', *(pdir / 'pairs').glob('*.npz')])
    return {str(p.relative_to(pdir)): sha256_file(p) for p in files}

def read_samples(pdir, dataset):
    return pd.read_csv(Path(pdir) / f'{dataset}_sample_splits.csv', dtype={'subject_id': str, 'identity_id': str, 'sample_id': str, 'partition': str})

def read_subject_splits(pdir):
    return pd.read_csv(Path(pdir) / 'MOLF_subject_splits.csv', dtype={'subject_id': str, 'partition': str})

def load_npz(path):
    with np.load(path, allow_pickle=False) as z: return {k: z[k] for k in z.files}

# ============================================================ Module A: expanded impostor pools
def _key(a, b, cross):
    a = np.asarray(a, np.int64); b = np.asarray(b, np.int64)
    if not cross: a, b = np.minimum(a, b), np.maximum(a, b)
    return a * 1_000_003 + b                                     # sample indices < 1e6

def eligible_counts(A, B, cross):
    """Number of eligible same-subject/different-finger and different-subject impostor pairs (within: unordered, cross: DB1 -> DB2)."""
    if cross:
        ns_a = A.groupby('subject_id').size(); ns_b = B.groupby('subject_id').size(); nf_a = A.groupby('identity_id').size(); nf_b = B.groupby('identity_id').size()
        same_subj = int((ns_a * ns_b.reindex(ns_a.index).fillna(0)).sum()); same_finger = int((nf_a * nf_b.reindex(nf_a.index).fillna(0)).sum())
        return same_subj - same_finger, int(len(A) * len(B) - same_subj)
    ns = A.groupby('subject_id').size(); nf = A.groupby('identity_id').size(); c2 = lambda n: n * (n - 1) // 2
    same_subj = int(c2(ns).sum()); same_finger = int(c2(nf).sum())
    return same_subj - same_finger, int(c2(len(A)) - same_subj)

def _enumerate_same(A, B, cross):
    out_a, out_b = [], []
    gb = {s: g for s, g in B.groupby('subject_id')}
    for s, ga in A.groupby('subject_id', sort=True):
        if cross:
            gbs = gb.get(s)
            if gbs is None: continue
            ia, ib = np.meshgrid(ga.sample_index.to_numpy(), gbs.sample_index.to_numpy(), indexing='ij'); fa, fb = np.meshgrid(ga.identity_id.to_numpy(), gbs.identity_id.to_numpy(), indexing='ij')
            m = fa != fb; out_a.append(ia[m]); out_b.append(ib[m])
        else:
            idx = ga.sample_index.to_numpy(); ids = ga.identity_id.to_numpy(); i, j = np.triu_indices(len(idx), 1); m = ids[i] != ids[j]
            out_a.append(np.minimum(idx[i[m]], idx[j[m]])); out_b.append(np.maximum(idx[i[m]], idx[j[m]]))
    return (np.concatenate(out_a), np.concatenate(out_b)) if out_a else (np.zeros(0, np.int64), np.zeros(0, np.int64))

def _sample_diff(A, B, need, seen, rng, cross, batch=200_000):
    a_idx = A.sample_index.to_numpy(np.int64); a_sub = A.subject_id.to_numpy(); b_idx = B.sample_index.to_numpy(np.int64); b_sub = B.subject_id.to_numpy()
    got_a, got_b, seen = [], [], set(seen); n = 0; rounds = 0
    while n < need:
        rounds += 1
        if rounds > 500: raise RuntimeError('cannot sample enough different-subject impostors')
        i = rng.integers(0, len(a_idx), batch); j = rng.integers(0, len(b_idx), batch); m = a_sub[i] != b_sub[j]; i, j = i[m], j[m]
        a, b = a_idx[i], b_idx[j]
        if not cross: a, b = np.minimum(a, b), np.maximum(a, b)
        k = _key(a, b, cross)
        k, first = np.unique(k, return_index=True); a, b = a[first], b[first]
        new = np.fromiter((x not in seen for x in k.tolist()), bool, len(k)); a, b, k = a[new], b[new], k[new]
        order = rng.permutation(len(k))[:need - n]; a, b, k = a[order], b[order], k[order]
        seen.update(k.tolist()); got_a.append(a); got_b.append(b); n += len(k)
    return np.concatenate(got_a), np.concatenate(got_b)

def build_expanded_pairs(A, B, trad, n_target, same_fraction, seed, cross):
    """A, B: DataFrames (sample_index, subject_id, identity_id) of the test images of dataset a / b. trad: traditional pair dict.
    Returns (pairs dict, info dict). Genuine pairs = traditional genuine pairs (unchanged order and direction); impostors = every
    traditional impostor + new unique impostors up to n_target (or all eligible), mixture same_fraction / 1 - same_fraction where feasible."""
    rng = np.random.default_rng(seed); lab = np.asarray(trad['label']).astype(np.int8); nt = np.asarray(trad['negative_type']).astype(np.int8)
    ta, tb = np.asarray(trad['index_a'], np.int64), np.asarray(trad['index_b'], np.int64)
    g = lab == 1; t_same = (lab == 0) & (nt == 1); t_diff = (lab == 0) & (nt == 2)
    el_same, el_diff = eligible_counts(A, B, cross); n_total = int(min(n_target, el_same + el_diff))
    want_same = int(round(same_fraction * n_total)); want_same = min(want_same, el_same); want_diff = min(n_total - want_same, el_diff); want_same = min(el_same, n_total - want_diff)
    want_same = max(want_same, int(t_same.sum())); want_diff = max(want_diff, int(t_diff.sum()))
    seen = set(_key(ta[~g], tb[~g], cross).tolist())
    # same-subject / different-finger: enumerate all eligible, drop the traditional ones, draw the rest
    sa, sb = _enumerate_same(A, B, cross); ks = _key(sa, sb, cross); keep = np.fromiter((k not in seen for k in ks.tolist()), bool, len(ks)); sa, sb, ks = sa[keep], sb[keep], ks[keep]
    take = rng.permutation(len(ks))[:max(0, want_same - int(t_same.sum()))]; sa, sb = sa[take], sb[take]; seen.update(ks[take].tolist())
    # different subject
    need_diff = max(0, want_diff - int(t_diff.sum()))
    if el_diff <= 3_000_000 and need_diff > 0:            # small pool: enumerate exactly
        Ad = A.sample_index.to_numpy(np.int64); Bd = B.sample_index.to_numpy(np.int64); As = A.subject_id.to_numpy(); Bs = B.subject_id.to_numpy()
        i, j = (np.triu_indices(len(Ad), 1) if not cross else np.meshgrid(np.arange(len(Ad)), np.arange(len(Bd)), indexing='ij'))
        i, j = np.ravel(i), np.ravel(j); m = As[i] != Bs[j]; da, db = Ad[i[m]], Bd[j[m]]
        if not cross: da, db = np.minimum(da, db), np.maximum(da, db)
        kd = _key(da, db, cross); keep = np.fromiter((k not in seen for k in kd.tolist()), bool, len(kd)); da, db = da[keep], db[keep]
        take = rng.permutation(len(da))[:need_diff]; da, db = da[take], db[take]
    elif need_diff > 0: da, db = _sample_diff(A, B, need_diff, seen, rng, cross)
    else: da, db = np.zeros(0, np.int64), np.zeros(0, np.int64)
    ia = np.r_[ta[g], ta[~g], sa, da]; ib = np.r_[tb[g], tb[~g], sb, db]
    label = np.r_[np.ones(int(g.sum()), np.int8), np.zeros(int((~g).sum()) + len(sa) + len(da), np.int8)]
    ntype = np.r_[np.zeros(int(g.sum()), np.int8), nt[~g], np.ones(len(sa), np.int8), np.full(len(da), 2, np.int8)]
    in_trad = np.r_[np.ones(len(ta), bool), np.zeros(len(sa) + len(da), bool)]
    P = {'index_a': ia, 'index_b': ib, 'label': label, 'negative_type': ntype, 'in_traditional': in_trad}
    n_imp = int((label == 0).sum()); n_s = int((ntype == 1).sum()); n_d = int((ntype == 2).sum())
    info = {'target_impostors': int(n_target), 'eligible_same_subject_different_finger': el_same, 'eligible_different_subject': el_diff,
            'impostors': n_imp, 'same_subject_different_finger': n_s, 'different_subject': n_d, 'same_subject_fraction_target': same_fraction,
            'same_subject_fraction_actual': n_s / max(n_imp, 1), 'genuine_pairs': int(g.sum()), 'traditional_impostors_included': int((~g).sum()),
            'all_eligible_used': n_imp >= el_same + el_diff, 'mixture_feasible': abs(n_s - round(same_fraction * n_imp)) <= 1}
    return P, info

def validate_pairs(P, A, B, cross, allowed_subjects, forbidden_subjects=(), trad=None):
    """Focused checks on a pair set; returns {check: bool}."""
    Ai = A.set_index('sample_index'); Bi = B.set_index('sample_index'); ia, ib = np.asarray(P['index_a']), np.asarray(P['index_b'])
    ok_idx = bool(np.isin(ia, Ai.index).all() and np.isin(ib, Bi.index).all())
    out = {'valid_indices': ok_idx}
    if not ok_idx: return out
    ida = Ai.identity_id.loc[ia].to_numpy(); idb = Bi.identity_id.loc[ib].to_numpy(); sa = Ai.subject_id.loc[ia].to_numpy(); sb = Bi.subject_id.loc[ib].to_numpy()
    lab = np.asarray(P['label']); nt = np.asarray(P['negative_type'])
    keys = _key(ia, ib, cross)
    out.update({'labels_match_identity': bool(np.all((ida == idb) == (lab == 1))),
                'same_subject_stratum_correct': bool(np.all((sa == sb)[nt == 1]) and np.all((ida != idb)[nt == 1])),
                'different_subject_stratum_correct': bool(np.all((sa != sb)[nt == 2])),
                'no_duplicates_including_reversed': bool(len(np.unique(keys)) == len(keys)),
                'no_self_pairs': bool(cross or np.all(ia != ib)),
                'subjects_in_allowed_partition': bool(set(sa) | set(sb) <= set(map(str, allowed_subjects))),
                'no_forbidden_subjects': bool(not ((set(sa) | set(sb)) & set(map(str, forbidden_subjects))))})
    if trad is not None:
        g = np.asarray(trad['label']) == 1; eg = np.asarray(P['label']) == 1
        out['genuine_pairs_identical_to_traditional'] = bool(np.array_equal(np.asarray(trad['index_a'])[g], ia[eg]) and np.array_equal(np.asarray(trad['index_b'])[g], ib[eg]))
        tk = set(_key(np.asarray(trad['index_a'])[~g], np.asarray(trad['index_b'])[~g], cross).tolist()); ek = set(keys[~eg].tolist())
        out['traditional_impostors_included'] = tk <= ek
    return out

def pair_manifest(P, A, B, pair_set, dataset_a, dataset_b):
    Ai = A.set_index('sample_index'); Bi = B.set_index('sample_index'); ia, ib = P['index_a'], P['index_b']
    return pd.DataFrame({'pair_set': pair_set, 'dataset_a': dataset_a, 'dataset_b': dataset_b, 'index_a': ia, 'index_b': ib,
                         'sample_id_a': Ai.sample_id.loc[ia].to_numpy(), 'sample_id_b': Bi.sample_id.loc[ib].to_numpy(),
                         'subject_a': Ai.subject_id.loc[ia].to_numpy(), 'subject_b': Bi.subject_id.loc[ib].to_numpy(),
                         'finger_a': Ai.identity_id.loc[ia].str.split('|').str[1:].str.join('|').to_numpy(), 'finger_b': Bi.identity_id.loc[ib].str.split('|').str[1:].str.join('|').to_numpy(),
                         'identity_a': Ai.identity_id.loc[ia].to_numpy(), 'identity_b': Bi.identity_id.loc[ib].to_numpy(),
                         'label': P['label'], 'negative_type': P['negative_type'], 'stratum': pd.Series(P['negative_type']).map(STRATUM_NAMES).to_numpy(), 'in_traditional': P['in_traditional']})

# ============================================================ scoring (same views as FPF01.evaluate)
def l2n(E): return E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)

def scoring_view(E, mus, dataset, scoring):
    """E: L2-normalised embeddings of `dataset`; mus: centering vectors saved by FPF01 (mu_src, mu_DB2_k*)."""
    if scoring == 'raw': return E
    if scoring == 'centered_source' or dataset == 'MOLF_DB1': mu = mus['mu_src']
    else: mu = mus[f"mu_DB2_k{int(scoring.rsplit('_k', 1)[1])}"]
    return l2n(np.asarray(E, np.float32) - np.asarray(mu, np.float32)[None, :])

def threshold_family(scoring): return 'raw' if scoring == 'raw' else 'centered'

# ============================================================ metrics (unweighted, Module A)
def roc_summary(y, s):
    fpr, tpr, thr = roc_curve(y, s); fnr = 1 - tpr; j = int(np.nanargmin(np.abs(fpr - fnr)))
    def tar_at(t):
        ok = np.flatnonzero(fpr <= t); return float(tpr[ok[-1]]) if len(ok) else 0.0
    return {'roc_auc': float(roc_auc_score(y, s)), 'eer': float((fpr[j] + fnr[j]) / 2), 'roc_tar_at_fmr_1pct': tar_at(1e-2), 'roc_tar_at_fmr_0_1pct': tar_at(1e-3)}

def frozen_threshold_counts(y, s, stratum, thr):
    acc = np.asarray(s) >= thr; y = np.asarray(y); st = np.asarray(stratum); out = {}
    g = y == 1; out['fnmr_at_frozen_threshold'] = (int((~acc & g).sum()), int(g.sum())); out['tar_at_frozen_threshold'] = (int((acc & g).sum()), int(g.sum()))
    for k, name in ((1, 'fmr_same_subject_different_finger'), (2, 'fmr_different_subject')):
        m = (y == 0) & (st == k); out[name] = (int((acc & m).sum()), int(m.sum()))
    m = y == 0; out['fmr_combined'] = (int((acc & m).sum()), int(m.sum()))
    return out

def expanded_rows(keys, y, s, stratum, thresholds, scoring):
    """Long rows for one (run, pair_set, scoring): every frozen-threshold metric for both rules + threshold-free ROC metrics."""
    fam = threshold_family(scoring); roc = roc_summary(y, s); rows = []; st = np.asarray(stratum); n_imp = max(int((np.asarray(y) == 0).sum()), 1)
    mix = f"same {100 * float(((np.asarray(y) == 0) & (st == 1)).sum()) / n_imp:.1f}% / different {100 * float(((np.asarray(y) == 0) & (st == 2)).sum()) / n_imp:.1f}%"
    for rule, thr in thresholds[fam].items():
        base = {**keys, 'evaluation_protocol': 'expanded', 'scoring': scoring, 'threshold_rule': rule, 'threshold_family': fam, 'threshold': float(thr), 'impostor_mixture': mix}
        for m, (num, den) in frozen_threshold_counts(y, s, stratum, thr).items():
            rows.append({**base, 'metric': m, 'value': num / den if den else np.nan, 'numerator': num, 'denominator': den, 'threshold_dependent': True,
                         'note': MIXTURE_NOTE if m == 'fmr_combined' else ('TAR at the frozen DB1 training threshold' if m.startswith('tar') else '')})
        for m, v in roc.items():
            rows.append({**base, 'metric': m, 'value': v, 'numerator': np.nan, 'denominator': np.nan, 'threshold_dependent': False,
                         'note': 'ROC-derived on this test set (not at the frozen threshold)' if m.startswith('roc_tar') else 'threshold-free; repeated under each threshold rule for joining'})
    return rows

# ============================================================ Module B: subject-cluster bootstrap
def subject_draws(n_subjects, n_boot, seed):
    """(n_boot, n_subjects) multiplicities m_i: N people drawn with replacement from the N test people."""
    rng = np.random.default_rng(seed); idx = rng.integers(0, n_subjects, (n_boot, n_subjects))
    M = np.zeros((n_boot, n_subjects), np.int32)
    for b in range(n_boot): M[b] = np.bincount(idx[b], minlength=n_subjects)
    return M

class PairStructure:
    """Fixed per (pair_set, protocol): subject positions of both endpoints, labels, strata, stratum proportions."""
    def __init__(self, label, stratum, subj_a, subj_b, subjects):
        pos = {s: i for i, s in enumerate(subjects)}
        self.y = np.asarray(label).astype(np.int8); self.st = np.asarray(stratum).astype(np.int8)
        self.a = np.array([pos[s] for s in subj_a]); self.b = np.array([pos[s] for s in subj_b])
        self.g = self.y == 1; self.s1 = (self.y == 0) & (self.st == 1); self.s2 = (self.y == 0) & (self.st == 2)
        if not np.all(self.a[self.g | self.s1] == self.b[self.g | self.s1]): raise ValueError('genuine / same-subject pairs must share the subject')
        n_imp = max(int(self.s1.sum() + self.s2.sum()), 1)
        self.p1 = float(self.s1.sum()) / n_imp; self.p2 = float(self.s2.sum()) / n_imp       # predefined (realised) stratum proportions, fixed
        self.n_subjects = len(subjects)
    def weights(self, M):
        M = np.asarray(M, np.float64); W = M[:, self.a].copy(); W[:, self.s2] *= M[:, self.b[self.s2]]; return W

class ScoreGrid:
    """Precomputed rank structure of one score vector, so weighted AUC / EER cost O(K) per draw."""
    def __init__(self, s, ps):
        s = np.asarray(s, np.float64); u, inv = np.unique(s, return_inverse=True); K = len(u); n = len(s)
        self.u = u; self.K = K
        self.Pm = sparse.csr_matrix((np.ones(n), (np.arange(n), inv)), shape=(n, K))   # pair -> unique-score group

def weighted_metrics(W, ps, grid, s, thresholds):
    """W: (c, n) pair weights. Returns {metric: (c,) array or {rule: array}} with NaN where undefined."""
    s = np.asarray(s); Wg = W[:, ps.g].sum(1); W1 = W[:, ps.s1].sum(1); W2 = W[:, ps.s2].sum(1)
    out = {'_valid_genuine': Wg > 0, '_valid_same': W1 > 0, '_valid_diff': W2 > 0}
    with np.errstate(invalid='ignore', divide='ignore'):
        for rule, thr in thresholds.items():
            acc = s >= thr
            fnmr = (W[:, ps.g] * ~acc[ps.g]).sum(1) / Wg; f1 = (W[:, ps.s1] * acc[ps.s1]).sum(1) / W1; f2 = (W[:, ps.s2] * acc[ps.s2]).sum(1) / W2
            parts = []
            if ps.p1 > 0: parts.append(ps.p1 * f1)
            if ps.p2 > 0: parts.append(ps.p2 * f2)
            comb = np.sum(parts, axis=0) if parts else np.full(len(W), np.nan)
            out[rule] = {'fnmr_at_frozen_threshold': np.where(Wg > 0, fnmr, np.nan), 'tar_at_frozen_threshold': np.where(Wg > 0, 1 - fnmr, np.nan),
                         'fmr_same_subject_different_finger': np.where(W1 > 0, f1, np.nan), 'fmr_different_subject': np.where(W2 > 0, f2, np.nan),
                         'fmr_combined': np.where(((W1 > 0) | (ps.p1 == 0)) & ((W2 > 0) | (ps.p2 == 0)), comb, np.nan)}
        # threshold-free: genuine weights normalised; impostor weights normalised within stratum and mixed with the fixed proportions
        G = np.zeros_like(W); G[:, ps.g] = W[:, ps.g] / Wg[:, None]
        I = np.zeros_like(W)
        if ps.p1 > 0: I[:, ps.s1] = ps.p1 * W[:, ps.s1] / W1[:, None]
        if ps.p2 > 0: I[:, ps.s2] = ps.p2 * W[:, ps.s2] / W2[:, None]
        G = np.nan_to_num(G); I = np.nan_to_num(I)
        gk = np.asarray(grid.Pm.T @ G.T).T; ik = np.asarray(grid.Pm.T @ I.T).T          # (c, K) weight per unique score, ascending
        below = np.cumsum(ik, 1) - ik
        auc = (gk * (below + 0.5 * ik)).sum(1) / np.maximum(ik.sum(1), 1e-300)
        tpr = np.cumsum(gk[:, ::-1], 1); fpr = np.cumsum(ik[:, ::-1], 1) / np.maximum(ik.sum(1, keepdims=True), 1e-300)   # accept s >= u_k, k descending
        tpr = np.c_[np.zeros(len(W)), tpr]; fpr = np.c_[np.zeros(len(W)), fpr]; fnr = 1 - tpr
        j = np.argmin(np.abs(fpr - fnr), 1); r = np.arange(len(W)); eer = (fpr[r, j] + fnr[r, j]) / 2
    valid_free = (Wg > 0) & ((W1 > 0) | (ps.p1 == 0)) & ((W2 > 0) | (ps.p2 == 0))
    out['roc_auc'] = np.where(valid_free, auc, np.nan); out['eer'] = np.where(valid_free, eer, np.nan)
    return out

BOOT_METRICS_THR = ('fmr_combined', 'fmr_same_subject_different_finger', 'fmr_different_subject', 'fnmr_at_frozen_threshold', 'tar_at_frozen_threshold')
BOOT_METRICS_FREE = ('roc_auc', 'eer')

def bootstrap_scores(M, ps, s, thresholds, chunk_elems=8_000_000):
    """Point estimate (all m_i = 1) and per-draw values for every metric. Returns ({(rule|None, metric): point}, {(rule|None, metric): (B,) draws})."""
    grid = ScoreGrid(s, ps); B = len(M); n = len(s); c = max(1, int(chunk_elems // max(n, 1)))
    point = weighted_metrics(ps.weights(np.ones((1, ps.n_subjects))), ps, grid, s, thresholds)
    pts = {(None, m): float(point[m][0]) for m in BOOT_METRICS_FREE}; pts.update({(r, m): float(point[r][m][0]) for r in thresholds for m in BOOT_METRICS_THR})
    draws = {k: np.empty(B) for k in pts}
    for i in range(0, B, c):
        out = weighted_metrics(ps.weights(M[i:i + c]), ps, grid, s, thresholds)
        for (r, m) in draws: draws[(r, m)][i:i + c] = out[m] if r is None else out[r][m]
    return pts, draws

def percentile_ci(values, level=0.95):
    v = np.asarray(values, float); ok = np.isfinite(v); a = (1 - level) / 2
    if ok.sum() == 0: return np.nan, np.nan, 0, len(v)
    lo, hi = np.percentile(v[ok], [100 * a, 100 * (1 - a)]); return float(lo), float(hi), int(ok.sum()), int((~ok).sum())

def ci_row(base, metric, point, draws, level, observed=None):
    lo, hi, nv, ni = percentile_ci(draws, level); flags = []
    if metric.startswith('fmr') and observed is not None and observed[0] == 0:
        flags.append('zero_false_matches_observed: the nonparametric bootstrap can give a degenerate [0, 0] interval; this is not evidence of zero population FMR')
    if nv < len(draws): flags.append(f'{ni} undefined draws excluded (missing stratum or class weight)')
    return {**base, 'metric': metric, 'point_estimate': point, 'ci_low': lo, 'ci_high': hi, 'confidence_level': level, 'n_bootstraps': len(draws), 'n_valid': nv, 'n_invalid': ni,
            'observed_numerator': observed[0] if observed else np.nan, 'observed_denominator': observed[1] if observed else np.nan,
            'degenerate_interval': bool(np.isfinite(lo) and lo == hi), 'flags': '; '.join(flags),
            'interval_note': 'subject-cluster percentile bootstrap; conditional on the fitted model, the sampled pair protocol and the chosen adaptation subjects '
                             '(excludes retraining and adaptation-subject selection uncertainty)'}

def diff_row(base, metric, variant_a, variant_b, pa, pb, da, db, level):
    d = np.asarray(da) - np.asarray(db); lo, hi, nv, ni = percentile_ci(d, level)
    return {**base, 'metric': metric, 'variant_a': variant_a, 'variant_b': variant_b, 'difference_definition': 'variant_a minus variant_b (same subject draws)',
            'point_difference': pa - pb, 'ci_low': lo, 'ci_high': hi, 'confidence_level': level, 'n_bootstraps': len(d), 'n_valid': nv, 'n_invalid': ni,
            'ci_excludes_zero': bool(np.isfinite(lo) and (lo > 0 or hi < 0)), 'note': 'descriptive interval; no p-value or significance test implied'}

def hand_check():
    """Small hand-checkable example of the subject-multiplicity weights (used by the notebooks' validation cells)."""
    subjects = ['A', 'B', 'C']
    # pairs: genuine A, genuine B, genuine C, same-subject A, different A-B, different B-C, different A-C
    y = np.array([1, 1, 1, 0, 0, 0, 0]); st = np.array([0, 0, 0, 1, 2, 2, 2]); sa = ['A', 'B', 'C', 'A', 'A', 'B', 'A']; sb = ['A', 'B', 'C', 'A', 'B', 'C', 'C']
    s = np.array([0.9, 0.4, 0.8, 0.7, 0.6, 0.2, 0.3]); thr = {'t': 0.5}
    ps = PairStructure(y, st, sa, sb, subjects); M = np.array([[2, 0, 1]])        # A drawn twice, B never, C once
    W = ps.weights(M)
    expected_W = np.array([2, 0, 1, 2, 0, 0, 2])                                  # genuine/same: m_i; different: m_i*m_j (A-B=0, B-C=0, A-C=2)
    out = weighted_metrics(W, ps, ScoreGrid(s, ps), s, thr)['t']
    # by hand: genuine weights A2(0.9 accept) C1(0.8 accept) -> FNMR 0; same A weight 2 score 0.7 accepted -> FMR_same 1;
    # different: only A-C weight 2 score 0.3 rejected -> FMR_diff 0; combined = 0.25*1 + 0.75*0 = 0.25 (pool proportions 1/4, 3/4)
    exp = {'fnmr_at_frozen_threshold': 0.0, 'fmr_same_subject_different_finger': 1.0, 'fmr_different_subject': 0.0, 'fmr_combined': 0.25}
    ok_w = bool(np.array_equal(W[0], expected_W)); ok_m = all(abs(out[k][0] - v) < 1e-12 for k, v in exp.items())
    # all m_i = 1 (the point estimate): FNMR 1/3 (B rejected); FMR_same 1/1; FMR_diff 1/3 (A-B accepted); combined 0.25*1 + 0.75*1/3 = 0.5 = pooled 2/4
    o2 = weighted_metrics(ps.weights(np.ones((1, 3))), ps, ScoreGrid(s, ps), s, thr)['t']
    ok_unit = all(abs(o2[k][0] - v) < 1e-12 for k, v in {'fnmr_at_frozen_threshold': 1 / 3, 'fmr_same_subject_different_finger': 1.0, 'fmr_different_subject': 1 / 3, 'fmr_combined': 0.5}.items())
    return {'weights_match_hand_example': ok_w, 'weighted_metrics_match_hand_example': ok_m, 'unit_weights_match_unweighted_counts': bool(ok_unit),
            'example': {'weights': W[0].tolist(), 'metrics': {k: float(out[k][0]) for k in exp}}}

In [ ]:
import sys, importlib; sys.path.insert(0, '/content')
import fpf_lib as L; importlib.reload(L)
import numpy as np, pandas as pd, json, time, os, gc, shutil, re, io
from pathlib import Path
ws = L.workspace()
for m in ('fpf_lib.py', 'fpf_tf.py', 'fpf_eval.py'): shutil.copy2(Path('/content') / m, ws['code'] / m)          # shared code for every FPF kernel
print('Workspace:', ws['root'], '| shared code ->', ws['code'])
ENV = L.env_info(); print(json.dumps(ENV, indent=1))

## 1. Configuration

In [ ]:
import ctypes, hashlib, struct, subprocess, zipfile, cv2, h5py, itertools
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
KERNEL = 'FPF00'; KDIR = L.kernel_dir(ws, KERNEL); FIG = KDIR / 'figures'; LOG = L.RunLog(KDIR / 'FPF00_run_log.csv')
for k in ('index', 'originals', 'images', 'templates', 'protocol'): ws[k].mkdir(parents=True, exist_ok=True)
RAW = Path('/content/fpf_raw')                                  # local extraction (fast; nothing generated outside output/ is kept)
FJFX_REPO = 'https://github.com/FingerJetFXOSE/FingerJetFXOSE.git'; FJFX_COMMIT = '1726ba08bf7f2137d2f861ac1ae124d5cd355eee'
FJFX_SRC = Path('/content/FingerJetFXOSE'); FJFX_SELFTEST_SHA256 = '336abe935f524ff81a8cc359b300e32a53f3646c060c027b3eea846aa4fe20e7'
DEFAULT_PPI = 500; IMAGE_SIZE = 224; WORKERS = max(2, os.cpu_count() or 2); TIMING_SAMPLES = 300

N_REPETITIONS = 3; BASE_SEED = 20261105
REP_SEEDS = [int(s) for s in np.random.SeedSequence(BASE_SEED).generate_state(N_REPETITIONS)]
SUBJECTS_PER_PARTITION = {'train': 60, 'validation': 15, 'adaptation': 5, 'test': 20}
TRAIN_IMPOSTOR_RATIO = 4; EVAL_IMPOSTOR_RATIO = 1; SAME_SUBJECT_IMPOSTOR_FRACTION = 0.25
MOLF_SESSION_RULE = 'impressions 1-2 = session 1, impressions 3-4 = session 2 (MOLF documentation: 4 impressions over two sessions)'
CONFIG = dict(kernel=KERNEL, datasets=L.DATASETS, sensors=L.SENSORS, casia='not used (corruptions)', archives=L.ARCHIVES, extractor='FingerJetFX OSE', fjfx_commit=FJFX_COMMIT,
              template_format='ISO/IEC 19794-2:2005', theta_convention='degrees, image frame (y down) = (360 - ISO angle) mod 360', type_codes='0 ending, 1 bifurcation, 2 other',
              quality='ISO minutia quality / 100', resolution='image ppi from the PNG pHYs chunk / BMP header / WSQ header, else 500', default_ppi=DEFAULT_PPI,
              image_representation='224x224 greyscale, aspect-preserving letterbox on white (cv2 INTER_AREA when shrinking) — identical to FPM02',
              repetitions=N_REPETITIONS, base_seed=BASE_SEED, repetition_seeds=REP_SEEDS, subjects_per_partition=SUBJECTS_PER_PARTITION,
              train_impostor_ratio=TRAIN_IMPOSTOR_RATIO, eval_impostor_ratio=EVAL_IMPOSTOR_RATIO, same_subject_impostor_fraction=SAME_SUBJECT_IMPOSTOR_FRACTION,
              molf_session_rule=MOLF_SESSION_RULE, fingerdos='all 60 subjects in every repetition (external test only)')
print(json.dumps(CONFIG, indent=2, default=str))

## 2. Extract the archives (local disk) and locate the image files

In [ ]:
def sh(cmd):
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if p.returncode != 0: raise RuntimeError(f'{cmd}\n{p.stdout[-2000:]}\n{p.stderr[-2000:]}')
    return p.stdout

def extract(archive, dst):
    marker = dst / '.extracted'
    if marker.is_file(): print(archive.name, 'already extracted'); return
    dst.mkdir(parents=True, exist_ok=True); local = L.local_copy(archive, '/content/fpf_archives'); t = L.timer()
    if archive.suffix.lower() == '.7z': sh(f'7z x -y -o"{dst}" "{local}"')
    else:
        tools = [f'unrar x -o+ -idq "{local}" "{dst}/"', f'bsdtar -xf "{local}" -C "{dst}"', f'7z x -y -o"{dst}" "{local}"']
        for c in tools:
            try: sh(c); break
            except Exception as exc: print('  extractor failed, trying the next one:', c.split()[0], str(exc)[:150])
        else: raise RuntimeError(f'cannot extract {archive}')
    for z in list(dst.rglob('*.zip')):                                         # archives that contain zipped DB folders
        with zipfile.ZipFile(z) as zz: zz.extractall(z.parent)
    marker.write_text('ok'); LOG.add(step='extract', archive=archive.name, status='ok', minutes=round(t() / 60, 2))

for key, name in L.ARCHIVES.items():
    src = ws['datasets'] / name
    if not src.is_file(): raise FileNotFoundError(f'{src} missing')
    extract(src, RAW / key)

IMG_EXT = {'.wsq', '.bmp', '.png', '.tif', '.tiff', '.jpg', '.jpeg', '.pgm'}
MOLF_RE = re.compile(r'^(?P<subject>\d+)_(?P<impression>\d+)_(?P<finger>\d+)\.(?P<ext>wsq|bmp|png|tif|tiff|jpg|jpeg|pgm)$', re.I)
FDOS_RE = re.compile(r'(?P<hand>LEFT|RIGHT)\s*HAND[\\/](?P<finger>THUMB|INDEX\s*FINGER|MIDDLE\s*FINGER)[\\/](?P<subject>\d{2})(?P<impression>\d{2})\.(?P<ext>bmp|png|tif|tiff)$', re.I)

def molf_db(path):
    # nearest folder that names exactly one sensor (the archive folder 'MOLF_DB1_and_2' names both and is skipped)
    for part in reversed([p.lower() for p in path.parts[:-1]]):
        db1 = ('db1' in part) or ('lumid' in part); db2 = ('db2' in part) or ('secu' in part)
        if db1 != db2: return 'MOLF_DB1' if db1 else 'MOLF_DB2'
    return None

FILES = {d: [] for d in L.DATASETS}; unmatched = []
for p in sorted((RAW / 'MOLF').rglob('*')):
    if not p.is_file() or p.suffix.lower() not in IMG_EXT: continue
    m = MOLF_RE.match(p.name); db = molf_db(p.relative_to(RAW / 'MOLF'))
    if m and db: FILES[db].append((p, m))
    else: unmatched.append(str(p.relative_to(RAW)))
for p in sorted((RAW / 'FingerDOS').rglob('*')):
    if not p.is_file() or p.suffix.lower() not in IMG_EXT: continue
    m = FDOS_RE.search(str(p.relative_to(RAW / 'FingerDOS')).replace('\\', '/'))
    if m: FILES['FingerDOS'].append((p, m))
    else: unmatched.append(str(p.relative_to(RAW)))
print({d: len(v) for d, v in FILES.items()}, '| unmatched image files:', len(unmatched), unmatched[:5])
for d in L.DATASETS:
    if len(FILES[d]) != L.EXPECTED_SAMPLES[d]:
        tree = sorted({str(p.relative_to(RAW).parent) for p, _ in FILES[d]})[:10]
        raise RuntimeError(f'{d}: found {len(FILES[d])} images, expected {L.EXPECTED_SAMPLES[d]}. Folders seen: {tree}. Unmatched examples: {unmatched[:10]}')

## 3. Decode, check and index (fixed order: subject, hand, finger, impression)

In [ ]:
if any(p.suffix.lower() == '.wsq' for d in FILES for p, _ in FILES[d]):     # WSQ codec only when WSQ files are present (the MOLF archive holds PNG)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wsq'], check=True); import wsq
from PIL import Image
FDOS_FINGER = {'THUMB': ('1', 'thumb'), 'INDEX FINGER': ('2', 'index'), 'MIDDLE FINGER': ('3', 'middle')}

def wsq_header(raw):
    txt = raw[:4096].decode('latin1', 'ignore'); out = {}
    for k in ('PIX_WIDTH', 'PIX_HEIGHT', 'PIX_DEPTH', 'PPI'):
        m = re.search(k + r'\s+(-?\d+)', txt)
        if m: out[k] = int(m.group(1))
    return out

def decode(path):
    raw = Path(path).read_bytes()
    if Path(path).suffix.lower() == '.wsq':
        img = np.array(Image.open(io.BytesIO(raw)).convert('L')); hd = wsq_header(raw)
        if 'PIX_WIDTH' in hd and (img.shape[1], img.shape[0]) != (hd['PIX_WIDTH'], hd['PIX_HEIGHT']): raise ValueError(f'{path}: decoded size {img.shape} != WSQ header {hd}')
        ppi = hd.get('PPI', -1); ppi = ppi if ppi and ppi > 0 else DEFAULT_PPI
    else:
        img = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_UNCHANGED)
        if img is None: raise ValueError(f'cannot decode {path}')
        if img.ndim == 3: img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY if img.shape[2] == 4 else cv2.COLOR_BGR2GRAY)
        if img.dtype != np.uint8: img = (img / 257).astype(np.uint8) if img.dtype == np.uint16 else img.astype(np.uint8)
        ppm = 0
        if raw[:2] == b'BM' and len(raw) > 42: ppm = struct.unpack('<i', raw[38:42])[0]
        elif raw[:8] == b'\x89PNG\r\n\x1a\n':
            k = raw.find(b'pHYs')
            if 0 < k < 4096 and raw[k + 12] == 1: ppm = struct.unpack('>I', raw[k + 4:k + 8])[0]    # unit 1 = metre
        ppi = int(round(ppm * 0.0254)) if ppm > 0 else DEFAULT_PPI
        ppi = ppi if 250 <= ppi <= 1500 else DEFAULT_PPI
    return np.ascontiguousarray(img, np.uint8), int(ppi)

def build_index(d):
    rows = []
    for p, m in FILES[d]:
        if d == 'FingerDOS':
            fname = re.sub(r'\s+', ' ', m['finger'].upper()); fid, fnm = FDOS_FINGER[fname]; hand = m['hand'][0].upper()
            rows.append(dict(subject_id=f'{int(m["subject"]):02d}', hand=hand, finger_id=fid, finger_name=f'{m["hand"].lower()} {fnm}', impression_id=f'{int(m["impression"]):02d}', session=0))
        else:
            imp = int(m['impression'])
            rows.append(dict(subject_id=f'{int(m["subject"]):03d}', hand='U', finger_id=f'{int(m["finger"]):02d}', finger_name=f'finger {int(m["finger"])}', impression_id=f'{imp:02d}', session=1 if imp <= 2 else 2))
        rows[-1]['source_file'] = str(p.relative_to(RAW))
    f = pd.DataFrame(rows).sort_values(['subject_id', 'hand', 'finger_id', 'impression_id']).reset_index(drop=True)
    f['identity_id'] = f.subject_id + '|' + f.hand + '|' + f.finger_id; f['sample_id'] = f.identity_id + '|' + f.impression_id
    if not f.sample_id.is_unique: raise RuntimeError(f'{d}: duplicate samples')
    f.insert(0, 'sample_index', np.arange(len(f), dtype=np.int64)); f.insert(0, 'dataset', d); f['sensor'] = L.SENSORS[d]
    return f

INDEX, DECODED = {}, {}
for d in L.DATASETS:
    f = build_index(d); t = L.timer()
    res = [decode(RAW / s) for s in tqdm(f.source_file, desc=f'{d} decode')]              # sequential (safe for every codec)
    DECODED[d] = [r[0] for r in res]; f['ppi'] = [r[1] for r in res]; f['height'] = [r[0].shape[0] for r in res]; f['width'] = [r[0].shape[1] for r in res]
    INDEX[d] = f; f.to_csv(ws['index'] / f'{d}_index.csv', index=False)
    print(d, len(f), 'images | subjects', f.subject_id.nunique(), '| fingers', f.identity_id.nunique(), '| impressions per finger', set(f.groupby('identity_id').size()),
          '| sizes', f.groupby(['width', 'height']).size().to_dict(), '| ppi', f.ppi.value_counts().to_dict(), f'| {t():.0f}s')
    LOG.add(step='decode_index', dataset=d, status='ok', images=len(f))

CHECKS = []
def check(name, ok, detail=''):
    CHECKS.append({'check': name, 'passed': bool(ok), 'detail': str(detail)}); print(('OK   ' if ok else 'FAIL ') + name, detail)
for d, (ns, nf, ni) in {'MOLF_DB1': (100, 10, 4), 'MOLF_DB2': (100, 10, 4), 'FingerDOS': (60, 6, 10)}.items():
    f = INDEX[d]; check(f'{d}: {ns} subjects', f.subject_id.nunique() == ns, f.subject_id.nunique())
    check(f'{d}: {nf} fingers per subject', set(f.groupby('subject_id').identity_id.nunique()) == {nf}, set(f.groupby('subject_id').identity_id.nunique()))
    check(f'{d}: {ni} impressions per finger', set(f.groupby('identity_id').size()) == {ni}, set(f.groupby('identity_id').size()))
check('MOLF DB1 and DB2: identical subjects', set(INDEX['MOLF_DB1'].subject_id) == set(INDEX['MOLF_DB2'].subject_id))
check('MOLF DB1 and DB2: identical finger identities', set(INDEX['MOLF_DB1'].identity_id) == set(INDEX['MOLF_DB2'].identity_id))
check('MOLF DB1 and DB2: same sample order', np.array_equal(INDEX['MOLF_DB1'].sample_id, INDEX['MOLF_DB2'].sample_id))
if not all(c['passed'] for c in CHECKS): raise RuntimeError('dataset structure checks failed (see above)')

## 4. Originals cache (lossless PNG bytes, so later kernels never need the WSQ codec)

In [ ]:
UTF8 = h5py.string_dtype('utf-8'); VU8 = h5py.vlen_dtype(np.dtype('uint8')); VF32 = h5py.vlen_dtype(np.dtype('float32'))
def write_meta(h, f, cols=('sample_id', 'identity_id', 'subject_id', 'hand', 'finger_id', 'finger_name', 'impression_id', 'source_file')):
    meta = h.create_group('metadata')
    for c in cols: meta.create_dataset(c, data=f[c].astype(str).to_numpy(object), dtype=UTF8)
for d in L.DATASETS:
    path = ws['originals'] / f'{d}_originals_png.h5'; f = INDEX[d]
    if path.is_file():
        with h5py.File(path, 'r') as h:
            if h.attrs.get('complete') and np.array_equal(h['metadata/sample_id'].asstr()[:], f.sample_id.to_numpy()): print(d, 'originals cache present'); continue
    local = Path('/content/fpf_local') / path.name; local.parent.mkdir(parents=True, exist_ok=True); t = L.timer()
    with h5py.File(local, 'w') as h:
        write_meta(h, f); h.create_dataset('png', (len(f),), dtype=VU8)
        for c in ('height', 'width', 'ppi'): h.create_dataset(c, data=f[c].to_numpy(np.int32))
        for i, img in enumerate(tqdm(DECODED[d], desc=f'{d} PNG')): h['png'][i] = np.frombuffer(cv2.imencode('.png', img, [cv2.IMWRITE_PNG_COMPRESSION, 3])[1].tobytes(), np.uint8)
        h.attrs['complete'] = True; h.attrs['note'] = 'lossless PNG of the decoded original greyscale image'
    L.copy_verified(local, path); LOG.add(step='originals_cache', dataset=d, status='ok', minutes=round(t() / 60, 2), mb=round(path.stat().st_size / 1e6, 1))

## 5. Build FingerJetFX (pinned commit, as FPM02) and self-test

In [ ]:
def run(cmd, cwd=None):
    p = subprocess.run(cmd, cwd=cwd, shell=True, text=True, capture_output=True)
    if p.returncode != 0: raise RuntimeError(f'{cmd}\n{p.stdout[-3000:]}\n{p.stderr[-3000:]}')
    return p.stdout
if not (FJFX_SRC / '.git').is_dir(): run(f'git clone -q {FJFX_REPO} {FJFX_SRC}')
run(f'git checkout -q {FJFX_COMMIT}', cwd=FJFX_SRC); proj = FJFX_SRC / 'FingerJetFXOSE'
run('bash runCMake.sh x64', cwd=proj); run(f'make -j{os.cpu_count()} FJFX', cwd=proj / 'build' / 'Linux-x86_64' / 'x64')
FJFX_LIB = proj / 'dist' / 'Linux-x86_64' / 'x64' / 'libFJFX.so'; shutil.copy2(FJFX_LIB, ws['templates'] / 'libFJFX.so')
FMD_ISO = 0x01010001; FMD_BUF = 34 + 256 * 6
_lib = ctypes.CDLL(str(FJFX_LIB)); _fjfx = _lib.fjfx_create_fmd_from_raw
_fjfx.argtypes = [ctypes.c_void_p, ctypes.c_ushort, ctypes.c_uint, ctypes.c_uint, ctypes.c_uint, ctypes.c_void_p, ctypes.POINTER(ctypes.c_uint)]; _fjfx.restype = ctypes.c_int
def extract_iso(gray, ppi):
    img = np.ascontiguousarray(gray, np.uint8); h, w = img.shape; buf = (ctypes.c_ubyte * FMD_BUF)(); size = ctypes.c_uint(FMD_BUF)
    err = _fjfx(img.ctypes.data_as(ctypes.c_void_p), int(ppi), h, w, FMD_ISO, ctypes.cast(buf, ctypes.c_void_p), ctypes.byref(size))
    return int(err), (bytes(buf[:size.value]) if err == 0 else b'')
def parse_iso(data):
    if data[:4] != b'FMR\x00': raise ValueError('not an ISO 19794-2 record')
    pos = 24; fq = data[pos + 2]; n = data[pos + 3]; pos += 4
    x = np.zeros(n, np.float32); y = np.zeros(n, np.float32); t = np.zeros(n, np.float32); k = np.zeros(n, np.uint8); q = np.zeros(n, np.float32)
    for i in range(n):
        a, b, ang, qq = struct.unpack('>HHBB', data[pos:pos + 6]); pos += 6; ty = a >> 14
        k[i] = 0 if ty == 1 else (1 if ty == 2 else 2); x[i] = a & 0x3FFF; y[i] = b & 0x3FFF; t[i] = (360.0 - ang * 360.0 / 256.0) % 360.0; q[i] = qq / 100.0
    return dict(x=x, y=y, theta=t, type=k, quality=q, finger_quality=fq)
def read_pgm(path):
    raw = Path(path).read_bytes(); parts = raw.split(maxsplit=4); w, h = int(parts[1]), int(parts[2]); return np.frombuffer(raw[-w * h:], np.uint8).reshape(h, w)
err, record = extract_iso(read_pgm(proj / 'libFJFX/samples/images-pgm/a001_03p.pgm'), 500); digest = hashlib.sha256(record).hexdigest(); n_ref = len(parse_iso(record)['x'])
print('Self-test: error', err, '| minutiae', n_ref, '| sha256', digest, '| bit-identical to FPM02' if digest == FJFX_SELFTEST_SHA256 else '| differs in bytes')
if err != 0 or abs(n_ref - 113) > 2: raise RuntimeError('FingerJetFX build does not reproduce the reference output')
LOG.add(step='build_fingerjetfx', status='ok', selftest_minutiae=n_ref, bit_identical=digest == FJFX_SELFTEST_SHA256)

## 6. FingerJetFX ISO templates for all three datasets (resumable)

In [ ]:
def process(args):
    img, ppi = args; t0 = time.perf_counter(); err, rec = extract_iso(img, ppi); return err, rec, 1000 * (time.perf_counter() - t0)
EMPTY = dict(x=np.zeros(0, np.float32), y=np.zeros(0, np.float32), theta=np.zeros(0, np.float32), type=np.zeros(0, np.uint8), quality=np.zeros(0, np.float32), finger_quality=0)
SUMMARY = []
for d in L.DATASETS:
    path = ws['templates'] / f'{d}_fingerjetfx_templates.h5'; f = INDEX[d]; n = len(f); t = L.timer()
    with h5py.File(path, 'a') as h:
        if 'minutiae' not in h:
            write_meta(h, f); g = h.create_group('minutiae')
            for c in ['x', 'y', 'theta', 'quality']: g.create_dataset(c, (n,), dtype=VF32)
            g.create_dataset('type', (n,), dtype=VU8); g.create_dataset('iso_template', (n,), dtype=VU8)
            for c in ['image_width', 'image_height', 'minutiae_count', 'ppi']: g.create_dataset(c, (n,), dtype='uint16')
            g.create_dataset('finger_quality', (n,), dtype='uint8'); g.create_dataset('extraction_error', (n,), dtype='int16'); g.create_dataset('extract_ms', (n,), dtype='float32')
            h.create_dataset('complete', (n,), dtype='bool', fillvalue=False); h.attrs.update({k: str(v) for k, v in CONFIG.items()})
        elif not np.array_equal(h['metadata/sample_id'].asstr()[:], f.sample_id.to_numpy()): raise RuntimeError(f'{d}: existing template file order mismatch')
        g = h['minutiae']; pending = np.flatnonzero(~h['complete'][:])
        with ThreadPoolExecutor(WORKERS) as pool:
            for s in tqdm(range(0, len(pending), 512), desc=f'{d} FingerJetFX'):
                idx = pending[s:s + 512]
                for i, (err, rec, ems) in zip(idx, pool.map(process, [(DECODED[d][i], f.ppi.iloc[i]) for i in idx])):
                    p = parse_iso(rec) if err == 0 else EMPTY
                    for c in ['x', 'y', 'theta', 'quality', 'type']: g[c][i] = p[c]
                    g['iso_template'][i] = np.frombuffer(rec, np.uint8); g['image_width'][i] = f.width.iloc[i]; g['image_height'][i] = f.height.iloc[i]; g['ppi'][i] = f.ppi.iloc[i]
                    g['minutiae_count'][i] = len(p['x']); g['finger_quality'][i] = p['finger_quality']; g['extraction_error'][i] = err; g['extract_ms'][i] = ems
                h['complete'][idx] = True; h.flush()
        counts = g['minutiae_count'][:].astype(int); errs = g['extraction_error'][:]
        types = np.concatenate([np.asarray(v) for v in g['type'][:] if len(v)]); q = np.concatenate([np.asarray(v) for v in g['quality'][:] if len(v)])
        SUMMARY.append({'dataset': d, 'sensor': L.SENSORS[d], 'images': n, 'extraction_failures': int((errs != 0).sum()), 'zero_minutiae': int((counts == 0).sum()),
                        'minutiae_mean': counts.mean(), 'minutiae_median': float(np.median(counts)), 'minutiae_p10': float(np.percentile(counts, 10)), 'minutiae_p90': float(np.percentile(counts, 90)),
                        'endings_pct': 100 * float((types == 0).mean()), 'bifurcations_pct': 100 * float((types == 1).mean()), 'minutia_quality_median': float(np.median(q)),
                        'extract_ms_median': float(np.median(g['extract_ms'][:])), 'minutes': t() / 60})
    LOG.add(step='templates', dataset=d, status='ok', minutes=round(t() / 60, 2))
template_summary = pd.DataFrame(SUMMARY); display(template_summary.round(2))

## 7. Standardised 224×224 images (letterbox on white, identical to FPM02)

In [ ]:
def letterbox(image):
    h, w = image.shape; s = min(IMAGE_SIZE / w, IMAGE_SIZE / h); nw, nh = max(1, round(w * s)), max(1, round(h * s))
    r = cv2.resize(image, (nw, nh), interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
    canvas = np.full((IMAGE_SIZE, IMAGE_SIZE), 255, np.uint8); x0 = (IMAGE_SIZE - nw) // 2; y0 = (IMAGE_SIZE - nh) // 2; canvas[y0:y0 + nh, x0:x0 + nw] = r; return canvas
for d in L.DATASETS:
    path = ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5'; f = INDEX[d]
    if path.is_file():
        with h5py.File(path, 'r') as h:
            if h['complete'][:].all() and np.array_equal(h['metadata/sample_id'].asstr()[:], f.sample_id.to_numpy()): print(d, 'image cache present'); continue
    local = Path('/content/fpf_local') / path.name; t = L.timer()
    with h5py.File(local, 'w') as h:
        write_meta(h, f); h.create_dataset('images', data=np.stack([letterbox(im) for im in DECODED[d]])[..., None], chunks=(1, IMAGE_SIZE, IMAGE_SIZE, 1), compression='lzf')
        h.create_dataset('complete', data=np.ones(len(f), bool)); h.attrs['representation'] = 'original_grayscale_aspect_preserving_letterbox_v1'
    L.copy_verified(local, path); LOG.add(step='image_cache', dataset=d, status='ok', minutes=round(t() / 60, 2))

## 8. Protocol: 3 repetitions, subject-disjoint MOLF split (same for DB1 and DB2) and pair files
`pairs/<set>_pairs.npz` holds `index_a`, `index_b` (sample indices in the datasets of `PAIR_SETS`), `label` (1 genuine) and `negative_type` (0 genuine, 1 same-subject impostor, 2 different-subject impostor).

In [ ]:
def genuine_within(f, rows):
    a, b = [], []
    for _, g in f.loc[rows].groupby('identity_id'):
        for i, j in itertools.combinations(g.sample_index.to_numpy(), 2): a.append(i); b.append(j)
    return np.array(a, np.int64), np.array(b, np.int64)

def impostors(fa, fb, rows_a, rows_b, n, rng, same_fraction, cross=False):
    # n impostor pairs: same_fraction from another finger of the same subject, the rest from different subjects; no duplicates, no self pairs
    A = fa.loc[rows_a]; B = fb.loc[rows_b]
    a_idx = A.sample_index.to_numpy(); a_sub = A.subject_id.to_numpy(); a_id = A.identity_id.to_numpy()
    b_idx = B.sample_index.to_numpy(); b_sub = B.subject_id.to_numpy(); b_id = B.identity_id.to_numpy()
    n_same = int(round(n * same_fraction)); seen = set(); out = []
    def key(i, j): return (i, j) if cross else (min(i, j), max(i, j))
    for want, same in ((n_same, True), (n - n_same, False)):
        got, tries = 0, 0
        while got < want:
            tries += 1
            if tries > 200 * max(want, 1): raise RuntimeError('cannot sample enough impostor pairs')
            k = rng.integers(len(a_idx))
            cand = np.flatnonzero((b_sub == a_sub[k]) & (b_id != a_id[k])) if same else np.flatnonzero(b_sub != a_sub[k])
            if len(cand) == 0: continue
            j = cand[rng.integers(len(cand))]; kk = key(int(a_idx[k]), int(b_idx[j]))
            if kk in seen or (not cross and kk[0] == kk[1]): continue
            seen.add(kk); out.append((kk[0], kk[1], 1 if same else 2)); got += 1
    o = np.array(out, np.int64); return o[:, 0], o[:, 1], o[:, 2]

def make_pairs(fa, fb, rows_a, rows_b, ratio, rng, cross=False):
    if cross:   # every DB1 impression x every DB2 impression of the same finger
        A = fa.loc[rows_a]; B = fb.loc[rows_b].groupby('identity_id').sample_index.apply(np.asarray).to_dict(); ga, gb = [], []
        for _, r in A.iterrows():
            for j in B[r.identity_id]: ga.append(r.sample_index); gb.append(j)
        ga, gb = np.array(ga, np.int64), np.array(gb, np.int64)
    else: ga, gb = genuine_within(fa, rows_a)
    ia, ib, nt = impostors(fa, fb, rows_a, rows_b, int(round(ratio * len(ga))), rng, SAME_SUBJECT_IMPOSTOR_FRACTION, cross)
    order = rng.permutation(len(ga) + len(ia))
    P = {'index_a': np.r_[ga, ia][order], 'index_b': np.r_[gb, ib][order], 'label': np.r_[np.ones(len(ga), np.int8), np.zeros(len(ia), np.int8)][order],
         'negative_type': np.r_[np.zeros(len(ga), np.int8), nt.astype(np.int8)][order]}
    return P

PROTO_SUMMARY = []
molf_subjects = sorted(INDEX['MOLF_DB1'].subject_id.unique())
for k, seed in enumerate(REP_SEEDS, 1):
    rep = f'rep_{k:02d}_seed_{seed}'; d = ws['protocol'] / rep; man_path = d / 'protocol_manifest.json'
    if man_path.is_file(): print(rep, 'protocol present'); PROTO_SUMMARY += json.loads(man_path.read_text())['summary']; continue
    (d / 'pairs').mkdir(parents=True, exist_ok=True); rng = np.random.default_rng(seed)
    perm = list(rng.permutation(molf_subjects)); part, a = {}, 0
    for p_, n_ in SUBJECTS_PER_PARTITION.items(): part.update({s: p_ for s in perm[a:a + n_]}); a += n_
    adapt = [s for s in perm if part[s] == 'adaptation']                         # nested budgets: first 1, first 2, all 5
    splits = pd.DataFrame({'subject_id': molf_subjects, 'partition': [part[s] for s in molf_subjects]})
    splits['adaptation_rank'] = [adapt.index(s) + 1 if s in adapt else 0 for s in molf_subjects]; splits.to_csv(d / 'MOLF_subject_splits.csv', index=False)
    for ds in L.DATASETS:
        s = INDEX[ds][['sample_index', 'sample_id', 'identity_id', 'subject_id']].copy()
        s['partition'] = 'external_test' if ds == 'FingerDOS' else s.subject_id.map(part)
        s.to_csv(d / f'{ds}_sample_splits.csv', index=False)
    rows = lambda ds, p_: INDEX[ds].index[INDEX[ds].subject_id.map(part) == p_] if ds != 'FingerDOS' else INDEX[ds].index
    spec = {'DB1_train': ('MOLF_DB1', 'MOLF_DB1', 'train', TRAIN_IMPOSTOR_RATIO, False), 'DB1_validation': ('MOLF_DB1', 'MOLF_DB1', 'validation', EVAL_IMPOSTOR_RATIO, False),
            'DB1_test': ('MOLF_DB1', 'MOLF_DB1', 'test', EVAL_IMPOSTOR_RATIO, False), 'DB2_test': ('MOLF_DB2', 'MOLF_DB2', 'test', EVAL_IMPOSTOR_RATIO, False),
            'DB1xDB2_test': ('MOLF_DB1', 'MOLF_DB2', 'test', EVAL_IMPOSTOR_RATIO, True), 'FingerDOS_test': ('FingerDOS', 'FingerDOS', 'external_test', EVAL_IMPOSTOR_RATIO, False)}
    summ = []
    for name, (da, db, p_, ratio, cross) in spec.items():
        P = make_pairs(INDEX[da], INDEX[db], rows(da, p_), rows(db, p_), ratio, np.random.default_rng([seed, list(spec).index(name)]), cross)
        np.savez_compressed(d / 'pairs' / f'{name}_pairs.npz', **P)
        summ.append({'repetition': rep, 'pair_set': name, 'dataset_a': da, 'dataset_b': db, 'partition': p_, 'subjects': len({*INDEX[da].loc[rows(da, p_)].subject_id}),
                     'images_a': len(rows(da, p_)), 'genuine_pairs': int(P['label'].sum()), 'impostor_pairs': int((P['label'] == 0).sum()),
                     'same_subject_impostors': int((P['negative_type'] == 1).sum()), 'different_subject_impostors': int((P['negative_type'] == 2).sum())})
    # leakage checks
    S = splits.set_index('subject_id').partition
    for p1, p2 in itertools.combinations(L.PARTITIONS, 2):
        if set(S[S == p1].index) & set(S[S == p2].index): raise RuntimeError(f'{rep}: subjects shared by {p1} and {p2}')
    for name, (da, db, p_, _, _) in spec.items():
        P = L.load_npz(d / 'pairs' / f'{name}_pairs.npz')
        for col, ds in (('index_a', da), ('index_b', db)):
            subj = set(INDEX[ds].subject_id.iloc[np.unique(P[col])])
            if ds != 'FingerDOS' and subj - set(S[S == p_].index): raise RuntimeError(f'{rep} {name}: subjects outside {p_}')
    man = {'repetition': rep, 'repetition_seed': seed, 'base_seed': BASE_SEED, 'subjects_per_partition': SUBJECTS_PER_PARTITION, 'adaptation_subjects_ranked': adapt,
           'pair_sets': {k_: list(v[:3]) for k_, v in spec.items()}, 'train_impostor_ratio': TRAIN_IMPOSTOR_RATIO, 'eval_impostor_ratio': EVAL_IMPOSTOR_RATIO,
           'same_subject_impostor_fraction': SAME_SUBJECT_IMPOSTOR_FRACTION, 'fingerdos': 'all 60 subjects', 'summary': summ}
    man_path.write_text(json.dumps(man, indent=2, default=str)); PROTO_SUMMARY += summ; LOG.add(step='protocol', repetition=rep, status='ok')
protocol_summary = pd.DataFrame(PROTO_SUMMARY); display(protocol_summary)
splits_all = pd.concat([pd.read_csv(ws['protocol'] / r / 'MOLF_subject_splits.csv').assign(repetition=r) for r in L.repetitions(ws)])
display(splits_all.groupby(['repetition', 'partition']).size().unstack())

## 9. Figures, extraction timing and workbook

In [ ]:
fig, axs = plt.subplots(3, 4, figsize=(15, 11))
for r_, d in enumerate(L.DATASETS):
    with h5py.File(ws['templates'] / f'{d}_fingerjetfx_templates.h5', 'r') as h, h5py.File(ws['images'] / f'{d}_standardized_grayscale_{IMAGE_SIZE}.h5', 'r') as hi:
        for c_, i in enumerate(np.random.default_rng(1).choice(len(INDEX[d]), 2, replace=False)):
            a = axs[r_, 2 * c_]; a.imshow(DECODED[d][i], cmap='gray'); x, y = h['minutiae/x'][i], h['minutiae/y'][i]; a.plot(x, y, 'r.', ms=3)
            a.set_title(f'{d} {INDEX[d].sample_id.iloc[i]}\n{DECODED[d][i].shape[1]}×{DECODED[d][i].shape[0]}, {len(x)} minutiae', fontsize=8); a.axis('off')
            b = axs[r_, 2 * c_ + 1]; b.imshow(hi['images'][i, ..., 0], cmap='gray', vmin=0, vmax=255); b.set_title('224×224 letterbox', fontsize=8); b.axis('off')
fig.suptitle('Originals with FingerJetFX minutiae, and the standardised model input', fontsize=10); L._save(fig, FIG / 'dataset_examples.png')
fig, axs = plt.subplots(1, 2, figsize=(12, 3.8))
for k, d in enumerate(L.DATASETS):
    with h5py.File(ws['templates'] / f'{d}_fingerjetfx_templates.h5', 'r') as h:
        axs[0].hist(h['minutiae/minutiae_count'][:], bins=40, alpha=.5, color=L.COLORS[k], label=d, density=True)
        axs[1].hist(np.concatenate([np.asarray(v) for v in h['minutiae/quality'][:] if len(v)]), bins=40, alpha=.5, color=L.COLORS[k], label=d, density=True)
axs[0].set_title('Minutiae per template'); axs[1].set_title('Minutia quality (0–1)')
for a in axs: a.legend(); a.grid(alpha=.3)
L._save(fig, FIG / 'minutiae_distributions.png')
dataset_summary = pd.DataFrame([{'dataset': d, 'sensor': L.SENSORS[d], 'images': len(f), 'subjects': f.subject_id.nunique(), 'fingers': f.identity_id.nunique(),
                                 'impressions_per_finger': int(f.groupby('identity_id').size().iloc[0]), 'image_sizes': json.dumps({f'{w}x{h}': int(n) for (w, h), n in f.groupby(['width', 'height']).size().items()}),
                                 'ppi': json.dumps(f.ppi.value_counts().to_dict())} for d, f in INDEX.items()])
display(dataset_summary)
L.write_workbook(KDIR / 'FPF00_Data_Preparation_Results.xlsx', {
    'Datasets': dataset_summary, 'Structure Checks': pd.DataFrame(CHECKS), 'FingerJetFX Templates': template_summary, 'Protocol Pairs': protocol_summary,
    'Subject Splits': splits_all, **{f'Index {d}': INDEX[d] for d in L.DATASETS}, 'Configuration': CONFIG, 'Run Log': LOG.frame(), 'Environment': ENV, 'Figure Index': L.figure_index(KDIR)})
print('FPF00 complete. Outputs in', KDIR)
for p in sorted(KDIR.rglob('*')):
    if p.is_file() and 'runs' not in p.parts: print(f'  {L.rel(ws, p)}  ({p.stat().st_size / 1e6:.1f} MB)')

## 10. Optional — expanded-impostor pair manifests (Module A input; standalone)

**Independent of sections 1–9:** run the setup cells at the top (install, mount, library, workspace) and then this section. It reads the index CSVs and the **traditional** protocol of every repetition and never rewrites them (their sha256 hashes are recorded in `protocol/<rep>/traditional_protocol_hashes.json` on the first run and verified on every later run).

For DB1_test, DB2_test, DB1xDB2_test and FingerDOS_test of every repetition:
* **genuine pairs = the traditional genuine pairs, unchanged** (same order and direction);
* **impostors = every traditional impostor + new unique impostors** up to `EXPANDED_IMPOSTOR_COUNT` (default 50,000), or all eligible pairs when fewer exist;
* predefined mixture **25% same subject / different finger, 75% different subject** where feasible (actual counts and proportions are saved);
* within-sensor pairs are unordered (no reversed duplicates); cross-sensor pairs keep the direction **DB1 → DB2**;
* only subjects of the test partition (MOLF) or all 60 FingerDOS subjects; adaptation subjects never enter MOLF test pairs.

Outputs (per repetition, `protocol/<rep>/expanded_pairs/`): `<set>_expanded_pairs.npz` (index_a, index_b, label, negative_type, in_traditional), `<set>_expanded_pairs_manifest.csv.gz` (dataset, subject, finger, image indices, label, impostor stratum), `<set>_expanded_manifest.json` (configuration, seed, counts, hashes). Cached files are reused only when their configuration hash and file hash match.

In [ ]:
RUN_EXPANDED_TEST = True
EXPANDED_IMPOSTOR_COUNT = 50000; EXPANDED_SAME_SUBJECT_FRACTION = 0.25; EXPANDED_SEED_TAG = 0xE5
if RUN_EXPANDED_TEST:
    import fpf_eval as EV; importlib.reload(EV)
    KDIR_X = L.kernel_dir(ws, 'FPF00'); EXP_CFG = {'eval_version': EV.EVAL_VERSION, 'impostor_count': EXPANDED_IMPOSTOR_COUNT, 'same_subject_fraction': EXPANDED_SAME_SUBJECT_FRACTION,
                                                   'seed_tag': EXPANDED_SEED_TAG, 'within_sensor': 'unordered pairs', 'cross_sensor': 'ordered DB1 -> DB2'}
    EXP_ROWS, EXP_CHECKS = [], []
    for rep in L.repetitions(ws):
        pdir = ws['protocol'] / rep; seed = L.rep_seed(rep); before = EV.protocol_hashes(pdir); hf = pdir / 'traditional_protocol_hashes.json'
        if hf.is_file():
            if json.loads(hf.read_text()) != before: raise RuntimeError(f'{rep}: traditional protocol files differ from their recorded hashes')
        else: hf.write_text(json.dumps(before, indent=2))
        spl = EV.read_subject_splits(pdir); adapt = spl[spl.partition == 'adaptation'].subject_id.tolist(); out = pdir / 'expanded_pairs'; out.mkdir(exist_ok=True)
        for k, name in enumerate(EV.TEST_SETS):
            da, db = EV.TEST_SET_DATASETS[name]; cross = da != db; part = 'external_test' if da == 'FingerDOS' else 'test'
            sa, sb = EV.read_samples(pdir, da), EV.read_samples(pdir, db); A, B = sa[sa.partition == part], sb[sb.partition == part]
            trad_file = pdir / 'pairs' / f'{name}_pairs.npz'; trad = EV.load_npz(trad_file)
            cfg = {**EXP_CFG, 'repetition': rep, 'pair_set': name, 'dataset_a': da, 'dataset_b': db, 'seed': [seed, k, EXPANDED_SEED_TAG],
                   'traditional_pairs_sha256': EV.sha256_file(trad_file), 'sample_splits_sha256': [EV.sha256_file(pdir / f'{x}_sample_splits.csv') for x in (da, db)]}
            cfg_hash = EV.json_hash(cfg); npz, mf = out / f'{name}_expanded_pairs.npz', out / f'{name}_expanded_manifest.json'
            man = json.loads(mf.read_text()) if mf.is_file() else {}
            if npz.is_file() and man.get('config_hash') == cfg_hash and man.get('pairs_sha256') == EV.sha256_file(npz):
                P = EV.load_npz(npz); info = man['counts']; print(rep, name, 'expanded pairs present (config and hash match)')
            else:
                P, info = EV.build_expanded_pairs(A, B, trad, EXPANDED_IMPOSTOR_COUNT, EXPANDED_SAME_SUBJECT_FRACTION, cfg['seed'], cross)
                np.savez_compressed(npz, **P); EV.pair_manifest(P, sa, sb, name, da, db).to_csv(out / f'{name}_expanded_pairs_manifest.csv.gz', index=False)
                mf.write_text(json.dumps({'config': cfg, 'config_hash': cfg_hash, 'pairs_sha256': EV.sha256_file(npz), 'counts': info, 'mixture_note': EV.MIXTURE_NOTE}, indent=2, default=str))
                print(rep, name, info)
            chk = EV.validate_pairs(P, sa, sb, cross, sorted(set(A.subject_id)), adapt if da != 'FingerDOS' else (), trad)
            EXP_CHECKS += [{'repetition': rep, 'pair_set': name, 'check': c, 'passed': v} for c, v in chk.items()]; EXP_ROWS.append({'repetition': rep, 'pair_set': name, **info})
        EXP_CHECKS.append({'repetition': rep, 'pair_set': 'all', 'check': 'traditional_protocol_unchanged', 'passed': EV.protocol_hashes(pdir) == before})
    expanded_summary = pd.DataFrame(EXP_ROWS); expanded_checks = pd.DataFrame(EXP_CHECKS); display(expanded_summary)
    print('expanded-pair checks:', 'ALL PASSED' if expanded_checks.passed.all() else expanded_checks[~expanded_checks.passed])
    if not expanded_checks.passed.all(): raise RuntimeError('expanded-pair validation failed')
    expanded_summary.to_csv(KDIR_X / 'FPF00_expanded_pairs_summary.csv', index=False); expanded_checks.to_csv(KDIR_X / 'FPF00_expanded_pairs_checks.csv', index=False)
    L.write_workbook(KDIR_X / 'FPF00_Expanded_Pairs.xlsx', {'Expanded Pair Counts': expanded_summary, 'Validation Checks': expanded_checks, 'Configuration': EXP_CFG})